In [ ]:
#| default_exp data_preprocessing

# Data Preprocessing

> Process your data, they say

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import zarr, numpy as np, pandas as pd, multiprocessing as mp, warnings, tqdm
from functools import partial
from collections import Counter
from pathlib import Path
from zarr.storage import ZipStore

def zarr_record_name(path):
    """Return a record name for either ``record.zarr.zip`` or a directory path."""
    name = Path(path).name
    if name.lower().endswith('.zarr.zip'):
        return name[:-len('.zarr.zip')]
    return Path(path).stem

def open_zarr_group(path, mode='r'):
    """Open a directory-backed Zarr group or consolidated ZipStore group."""
    if Path(path).suffix.lower() == '.zip':
        store = ZipStore(str(path), mode=mode, compression=0, allowZip64=True)
        return zarr.open_consolidated(store, mode=mode)
    return zarr.open(path, mode=mode)

def close_zarr_group(root_grp):
    """Close a group's backing store when it exposes a close method."""
    close = getattr(getattr(root_grp, 'store', None), 'close', None)
    if close is not None:
        close()

In [ ]:
#| export
def interpolate_nan_clip(x_in, physiological_range_clip=None, percentile_clip=None, return_mask_only=False):
    """
    Function to clip outliers based on percentiles or physiological range and then interpolate nearby values
    """
    x = np.copy(x_in)
    if physiological_range_clip is not None:
        # if a physiological range, clip, set to nan, and interpolate nearby values
        assert len(physiological_range_clip) == 2, "physiological_range_clip expects a tuple or list of 1 or 2 values. Supply none to clip only one end."
        max_ = physiological_range_clip[1] if physiological_range_clip[1] is not None else None
        min_ = physiological_range_clip[0] if physiological_range_clip[0] is not None else None
        if max_ is not None:
            x[x>=max_] = np.nan
        if min_ is not None:
            x[x<=min_] = np.nan
    if percentile_clip is not None:
        # if percentiles, clip at percentiles, set to nana, and interpolate nearby
        assert len(percentile_clip) == 2, "percentile_clip expects a tuple or list of 1 or 2 values. Supply none to clip only one end."
        max_ = np.quantile(x, q=percentile_clip[1]) if percentile_clip[1] is not None else None
        min_ = np.quantile(x, q=percentile_clip[0]) if percentile_clip[0] is not None else None
        if max_ is not None:
            x[x>=max_] = np.nan
        if min_ is not None:
            x[x<=min_] = np.nan
    mask = np.isnan(x)
    if return_mask_only:
        return mask
    if all(mask):
        return np.zeros_like(x) # return all zeros if all nan
    else:
        x[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), x[~mask])
        return x

In [ ]:
#| export
def check_signal_not_constant(channel_data, constant_tolerance=1.0):
    """
    Function to check if a signal is NOT constant. This works for NaNs and constant values.
    """
    if len(channel_data) == 0:
        return False
    _, counts = np.unique(channel_data, return_counts=True)
    max_ratio = np.max(counts) / len(channel_data)
    return max_ratio <= constant_tolerance # if more than half the signal is the same value, return False
    
def calculate_samples(zarr_file, channels, frequency, sample_seq_len_sec, stride_sec, start_offset_sec=None, max_seq_len_sec=None, include_partial_samples=True, require_all_channels=True, constant_nan_tolerance=1.0, min_seq_len_sec=None, constant_channels=None):
    """
    Function to create a dataframe of samples and their sequence indices
    """
    if start_offset_sec is None:
        start_offset_sec = 0
    if min_seq_len_sec is None:
        min_seq_len_sec = 0
    if constant_channels is None:
        constant_channels = []
    start_offset = start_offset_sec * frequency
    if max_seq_len_sec is not None:
        assert max_seq_len_sec >= sample_seq_len_sec, "The maximum sequence length should be >= the sample sequence length. The maximum sequence length is the end cutoff point of a sample. A sample cannot be longer than that value."
    if min_seq_len_sec is not None and max_seq_len_sec is not None:
        assert min_seq_len_sec <= max_seq_len_sec, "The minimum sequence length should be <= the max sequence length. The minimum sequence length is the start cutoff point of a sample. A sample cannot be shorter than that value."
    sample_seq_len = sample_seq_len_sec * frequency

    stride = stride_sec * frequency
    root_grp = open_zarr_group(zarr_file)
    updated_channels = []
    avail_channels = list(root_grp.array_keys())
    if all(isinstance(i, list) for i in channels):
        for p in channels:
            updated_channels.append(next((x for x in p if x in avail_channels), None))
    else:
        updated_channels = [p if p in avail_channels else None for p in channels]
    reason = 0
    if (None not in updated_channels) or (not require_all_channels and any(updated_channels)):
        # all channels are present or at least one channel is present and require_all_channels is False
        ## all channels should be the same length
        if 'header' in root_grp.attrs:
            duration = int(root_grp.attrs['header']['Duration']) # duration in seconds
        else:
            duration = int(root_grp.attrs['Duration']) # duration in seconds
        if duration == -1:
            warnings.warn(f"Duration is -1 for file {zarr_file}. Inferring from signal...")
            test_channel = next((i for i in updated_channels if i is not None))
            signal_frequency = root_grp[test_channel].attrs.asdict().get('signal_header', None).get('sample_frequency', None)
            if signal_frequency is None:
                duration = len(root_grp[test_channel][:]) // frequency
            else:
                duration = len(root_grp[test_channel][:]) // signal_frequency
        if duration > (start_offset_sec + min_seq_len_sec):
            if include_partial_samples:
                # if max_seq_len_sec is not None, use that, otherwise use the duration
                max_seq_len = max_seq_len_sec*frequency if max_seq_len_sec is not None else duration*frequency+sample_seq_len-1
            else:
                max_seq_len =  max_seq_len_sec*frequency if max_seq_len_sec is not None and max_seq_len_sec < duration else duration*frequency
            sample_indices = [(i, i+sample_seq_len) 
                    for i in range(start_offset, max_seq_len+start_offset, stride) 
                    if (i-start_offset)+sample_seq_len <= max_seq_len]
            sample_index_df = pd.DataFrame([{'start_idx':i[0], 'end_idx':i[1]} for i in sample_indices])
            sample_index_df['file'] = zarr_file
            if not sample_index_df.empty:
                exclude_channels = []
                for channel in updated_channels:
                    if channel is not None and channel not in constant_channels:
                        channel_data = root_grp[channel][:]
                        signal_header = root_grp[channel].attrs.asdict()#.get('signal_header', None)
                        if 'signal_header' in signal_header:
                            signal_header = signal_header['signal_header']
                            channel_frequency = signal_header.get('sample_frequency', None)
                        elif 'sampling_frequency' in signal_header:
                            channel_frequency = signal_header['sampling_frequency']
                        else:
                            warnings.warn(f"Channel {channel} has no signal header for file {zarr_file}. Skipping missingness and constant checks.")
                            exclude_channels.append(channel)
                            continue
                        if channel_frequency is None:
                            warnings.warn(f"Channel {channel} has no sample frequency for file {zarr_file}. Setting to passed frequency {frequency}.")
                            channel_frequency = frequency
                        sample_index_df['channel_frequency'] = channel_frequency
                        sample_index_df['channel_frequency_start_idx'] = (sample_index_df['start_idx'] / frequency * channel_frequency).astype(int)
                        sample_index_df['channel_frequency_end_idx'] = (sample_index_df['end_idx'] / frequency * channel_frequency).astype(int)

                        start_indices = sample_index_df['channel_frequency_start_idx'].values
                        end_indices = sample_index_df['channel_frequency_end_idx'].values
                        not_constant_results = []

                        for start_idx, end_idx in zip(start_indices, end_indices):
                            segment = channel_data[start_idx:end_idx]
                            not_constant_results.append(check_signal_not_constant(segment, constant_tolerance=constant_nan_tolerance))
                        
                        sample_index_df[f'{channel}_not_constant_nan'] = not_constant_results
                        sample_index_df.drop(columns=['channel_frequency_start_idx', 'channel_frequency_end_idx', 'channel_frequency'], inplace=True)
                # the maximum channel nan sum should be less than the nan tolerance
                sample_index_df['all_channels_not_constant_nan'] = sample_index_df[[f'{channel}_not_constant_nan' for channel in updated_channels if channel is not None and channel not in exclude_channels and channel not in constant_channels]].all(axis=1)
                sample_index_df = sample_index_df.loc[(sample_index_df['all_channels_not_constant_nan'] == True)]
                if sample_index_df.empty:
                    # all constant
                    close_zarr_group(root_grp)
                    return None, 4        
                sample_index_df['n_samples'] = len(sample_indices)
                close_zarr_group(root_grp)
                return sample_index_df, reason
            else:
                # empty df
                close_zarr_group(root_grp)
                return None, 3
        else:
            # start after duration
            close_zarr_group(root_grp)
            return None, 2
    else:
        # missing channels
        close_zarr_group(root_grp)
        return None, 1

def calculate_samples_mp(zarr_files, channels, frequency, sample_seq_len_sec, stride_sec, start_offset_sec = None, max_seq_len_sec=None, include_partial_samples=True, constant_nan_tolerance=1.0, require_all_channels=True, min_seq_len_sec=None, constant_channels=None, n_processes=None):
    """
    Multiprocessing function to generate samples
    """
    final_df = pd.DataFrame(columns=['file', 'start_idx','end_idx','n_samples'])
    reason_tracker = {0:'valid', 1:'missing channels', 2:'start after duration', 3:'empty df', 4:'nan filters'}
    reasons = []
    total_samples = 0
    with mp.Pool(processes=n_processes) as pool:
        f = partial(calculate_samples, channels=channels, frequency=frequency, start_offset_sec=start_offset_sec, max_seq_len_sec=max_seq_len_sec, sample_seq_len_sec=sample_seq_len_sec, stride_sec=stride_sec, include_partial_samples=include_partial_samples, constant_nan_tolerance=constant_nan_tolerance, require_all_channels=require_all_channels, min_seq_len_sec=min_seq_len_sec, constant_channels=constant_channels)
        # Use tqdm to show progress
        results = list(tqdm.tqdm(
            pool.imap_unordered(f, zarr_files),
            total=len(zarr_files),
            desc="Processing files"
        ))
        pool.close()
        pool.join()
    for result, reason in results:
        final_df = pd.concat([final_df, result])
        reasons.append(reason)
    final_df.reset_index(drop=True, inplace=True)
    total_samples = len(final_df)
    print('REMOVAL REASONS')
    print({reason_tracker[k]:v for k,v in Counter(reasons).items()})
    return final_df, total_samples

In [ ]:
#| export
def calculate_samples_forecast(zarr_file, outcome_start_times, outcome_durations, outcome_vals, channels, forecast_window_sec, frequency, sample_seq_len_sec, require_all_channels=True, constant_nan_tolerance=1.0, infer_forecast_windows=True, sample_frequency_key='sampling_frequency'):
    """
    Function to create a dataframe of samples and their sequence indices
    """
    if len(outcome_start_times) != len(outcome_vals):
        raise ValueError("outcome_start_times and outcome_vals must have same length")
    sample_seq_len = int(sample_seq_len_sec * frequency)
    outcome_start_idxs = [int(t * frequency) for t in outcome_start_times]
    outcome_end_idxs = [s + int(e * frequency) for s,e in zip(outcome_start_idxs, outcome_durations)]
    forecast_offsets = [int(f * frequency) for f in forecast_window_sec]
    
    root_grp = open_zarr_group(zarr_file)

    avail_channels = list(root_grp.array_keys())
    removal_reasons = {}
    
    updated_channels = []
    for ch in channels:
        if isinstance(ch, list):
            # Find first available channel from alternatives
            selected = next((x for x in ch if x in avail_channels), None)
        else:
            selected = ch if ch in avail_channels else None
        updated_channels.append(selected)

    # Check channel requirements
    valid_channels = [ch for ch in updated_channels if ch is not None]
    if require_all_channels and len(valid_channels) != len(channels):
        #missing = [ch for ch in channels if ch not in avail_channels]
        #raise ValueError(f"Required channels not available: {missing}")
        removal_reasons['1'] = 1 # missing channels
        close_zarr_group(root_grp)
        return None, removal_reasons
    elif not valid_channels:
        removal_reasons['2'] = 1 # no valid channels
        close_zarr_group(root_grp)
        return None, removal_reasons
    
    sample_dfs = []
    for forecast_sec, forecast_offset in zip(forecast_window_sec, forecast_offsets):
        end_idxs = [outcome_idx - forecast_offset for outcome_idx in outcome_start_idxs]
        start_idxs = [end_idx - sample_seq_len for end_idx in end_idxs]
        df = pd.DataFrame({
            'start_idx': start_idxs,
            'end_idx': end_idxs,
            'file_path': zarr_file,
            'outcome_start_idx': outcome_start_idxs,
            'outcome_end_idx': outcome_end_idxs,
            'outcome_duration_sec': outcome_durations,
            f'outcome_val_{forecast_sec}sec': outcome_vals
        })
        sample_dfs.append(df)
    sample_index_df = pd.concat(sample_dfs, ignore_index=True)
    agg_dict = {col: 'first' for col in sample_index_df.columns 
                if col not in ['start_idx', 'end_idx', 'file_path']}
    sample_index_df = sample_index_df.groupby(['start_idx', 'end_idx', 'file_path']).agg(agg_dict).reset_index()
    total_before = len(sample_index_df)

    data_length = len(root_grp[valid_channels[0]])
    valid_mask = (sample_index_df['start_idx'] >= 0) & (sample_index_df['end_idx'] <= data_length)
    sample_index_df = sample_index_df.loc[valid_mask]
    removal_reasons['3'] = (total_before - len(sample_index_df)) # number of samples removed due to invalid indices
    
    if infer_forecast_windows:
        for forecast_offset, forecast_sec in zip(forecast_offsets, forecast_window_sec):
            # check if the forecast window is within the outcome window, else assign to 0
            ## note that to force a 0, end outcome idx must be less than start outcome idx
            forecast_start_idxs = sample_index_df['end_idx'] + forecast_offset
            inferred_ = (forecast_start_idxs >= sample_index_df['outcome_start_idx']) & (forecast_start_idxs <= sample_index_df['outcome_end_idx'])
            sample_index_df[f'outcome_val_{forecast_sec}sec'] = sample_index_df[f'outcome_val_{forecast_sec}sec'].combine_first(inferred_.astype(int))

    if not sample_index_df.empty:
        for channel in updated_channels:
            if channel is not None:
                channel_data = root_grp[channel][:]
                channel_frequency = root_grp[channel].attrs.asdict().get(sample_frequency_key, None)
                if channel_frequency is None:
                    warnings.warn(f"Channel {channel} has no sampling frequency attribute in the zarr file. Setting to {frequency} Hz")
                    channel_frequency = frequency
                sample_index_df['channel_frequency'] = channel_frequency
                sample_index_df['channel_frequency_start_idx'] = (sample_index_df['start_idx'] / frequency * channel_frequency).astype(int)
                sample_index_df['channel_frequency_end_idx'] = (sample_index_df['end_idx'] / frequency * channel_frequency).astype(int)

                start_indices = sample_index_df['channel_frequency_start_idx'].values
                end_indices = sample_index_df['channel_frequency_end_idx'].values
                not_constant_results = []

                for start_idx, end_idx in zip(start_indices, end_indices):
                    segment = channel_data[start_idx:end_idx]
                    not_constant_results.append(check_signal_not_constant(segment, constant_tolerance=constant_nan_tolerance))
                sample_index_df[f'{channel}_not_constant_nan'] = not_constant_results
                sample_index_df.drop(columns=['channel_frequency_start_idx', 'channel_frequency_end_idx', 'channel_frequency'], inplace=True)
        
        # the maximum channel nan sum should be less than the nan tolerance
        sample_index_df['all_channels_not_constant_nan'] = sample_index_df[[f'{channel}_not_constant_nan' for channel in updated_channels if channel is not None]].all(axis=1)
        total_before = len(sample_index_df)
        sample_index_df = sample_index_df.loc[(sample_index_df['all_channels_not_constant_nan'] == True)]
        removal_reasons['5'] = (total_before - len(sample_index_df)) # number of samples removed due to nan or constant signal
                    
        sample_index_df['n_samples'] = 1
        close_zarr_group(root_grp)
        return sample_index_df, removal_reasons
    else:
        removal_reasons['6'] = 1 # no valid samples
        close_zarr_group(root_grp)
        return None, removal_reasons
        
def calculate_samples_forecast_mp(outcome_df, file_col, outcome_val_col, outcome_time_col, outcome_duration_col, forecast_window_sec, channels, frequency, sample_seq_len_sec, constant_nan_tolerance=1.0, require_all_channels=True, infer_forecast_windows=True, sample_frequency_key='sampling_frequency', n_processes=None):
    """
    Multiprocessing function to generate samples
    """
    args = outcome_df.groupby(file_col, as_index=False).agg({outcome_time_col:list, outcome_duration_col:list, outcome_val_col:list})
    args = args.to_dict(orient='records')
    args = list(map(lambda v: tuple(v.values()), args))
    removal_reasons = {"1":0, "2":0, "3":0, "4":0, "5":0}
    with mp.Pool(processes=n_processes) as pool:
        f = partial(calculate_samples_forecast, channels=channels, forecast_window_sec=forecast_window_sec, frequency=frequency, sample_seq_len_sec=sample_seq_len_sec, constant_nan_tolerance=constant_nan_tolerance, require_all_channels=require_all_channels, infer_forecast_windows=infer_forecast_windows, sample_frequency_key=sample_frequency_key)
        results = pool.starmap(f, tqdm.tqdm(args, total=len(args), desc="Processing files"))
        pool.close()
        pool.join()
    samples = []
    for result in results:
        if result[0] is not None:
            samples.append(result[0])
        for k, v in result[1].items():
            removal_reasons[k] = removal_reasons.get(k, 0) + v
    if samples:
        final_df = pd.concat(samples, ignore_index=True)
    else:
        final_df = pd.DataFrame()
    total_samples = len(final_df)
    return final_df, total_samples, removal_reasons

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()